# build_epss_distribution
Bronze EPSS **전량**(필터링 없이) → 하루 1행짜리 `silver_epss_distribution`.
개별 CVE의 EPSS가 아니라 **모델 전체가 재조정됐는지**를 감시하는 테이블이다 — 관심 CVE들만 보면 "덜 위험해졌다"고 오판할 수 있는 상황(모델 버전 변경 등)을 구분해준다(02_Bronze_Silver_저장구조.md §3.2 7번 항목).

`is_systemic`은 **전일 대비** 판단이므로, 어제 분포 행이 아직 없으면(최초 실행 등) 판정 자체가 불가능 → `is_systemic = NULL`(미판정)로 남긴다. `False`(판정했는데 아님)와 `NULL`(판정 불가)을 혼동하지 않는다 — 프로젝트 전반의 `NULL ≠ FALSE` 원칙(03_Gold_Schema.md §3.7)을 Silver 단계부터 지킨다.

In [ ]:
%run 00_silver_common

In [ ]:
import re
from datetime import datetime as _dt, timedelta as _timedelta

epss_dir = f"{BRONZE_EPSS_PATH}/{snapshot_date}"

header_line = ""
try:
    header_rdd = spark.sparkContext.textFile(f"{epss_dir}/*.csv.gz")
    header_line = " ".join(header_rdd.filter(lambda l: l.startswith("#")).take(5))
except Exception:
    pass

def parse_header_field(text: str, key: str):
    m = re.search(rf"{key}\s*:\s*([^,\s]+)", text)
    return m.group(1) if m else None

resolved_snapshot_date = parse_header_field(header_line, "score_date") or snapshot_date
resolved_model_version = parse_header_field(header_line, "model_version")

raw_df = spark.read.option("header", "true").option("comment", "#").csv(f"{epss_dir}/*.csv.gz")
score_col = coalesce_existing(raw_df, ["epss"]).cast(DoubleType())
raw_df = raw_df.withColumn("_score", score_col).filter(F.col("_score").isNotNull())


In [ ]:
agg_row = raw_df.select(
    F.count("*").alias("total_cve_count"),
    F.mean("_score").alias("mean_score"),
    F.expr("percentile_approx(_score, 0.5)").alias("median_score"),
    F.expr("percentile_approx(_score, 0.5)").alias("p50"),
    F.expr("percentile_approx(_score, 0.75)").alias("p75"),
    F.expr("percentile_approx(_score, 0.9)").alias("p90"),
    F.expr("percentile_approx(_score, 0.95)").alias("p95"),
    F.expr("percentile_approx(_score, 0.99)").alias("p99"),
    F.max("_score").alias("max_score"),
    F.sum(F.when(F.col("_score") > 0.01, 1).otherwise(0)).alias("score_gt_001_count"),
    F.sum(F.when(F.col("_score") > 0.1, 1).otherwise(0)).alias("score_gt_01_count"),
    F.sum(F.when(F.col("_score") > 0.5, 1).otherwise(0)).alias("score_gt_05_count"),
).first()


In [ ]:
prev_date = (_dt.strptime(resolved_snapshot_date, "%Y-%m-%d") - _timedelta(days=1)).strftime("%Y-%m-%d")

prev_row = None
if table_exists(TBL["epss_distribution"]):
    prev_collected = spark.table(TBL["epss_distribution"]).filter(F.col("snapshot_date") == prev_date).collect()
    prev_row = prev_collected[0] if prev_collected else None

def pct_change(new, old):
    if old in (None, 0) or new is None:
        return None
    return abs(new - old) / abs(old)

is_systemic = None
reasons = []
if prev_row is not None:
    is_systemic = False
    if resolved_model_version != prev_row["model_version"]:
        is_systemic = True
        reasons.append(f"MODEL_VERSION_CHANGED({prev_row['model_version']}->{resolved_model_version})")
    mean_chg = pct_change(agg_row["mean_score"], prev_row["mean_score"])
    if mean_chg is not None and mean_chg > 0.2:
        is_systemic = True
        reasons.append(f"MEAN_SHIFT({mean_chg:.2%})")
    p95_chg = pct_change(agg_row["p95"], prev_row["p95"])
    if p95_chg is not None and p95_chg > 0.2:
        is_systemic = True
        reasons.append(f"P95_SHIFT({p95_chg:.2%})")
    count_chg = pct_change(agg_row["score_gt_01_count"], prev_row["score_gt_01_count"])
    if count_chg is not None and count_chg > 0.3:
        is_systemic = True
        reasons.append(f"COUNT_SHIFT({count_chg:.2%})")


In [ ]:
dist_row = {
    "snapshot_date": resolved_snapshot_date,
    "model_version": resolved_model_version,
    "total_cve_count": int(agg_row["total_cve_count"]),
    "mean_score": agg_row["mean_score"],
    "median_score": agg_row["median_score"],
    "p50": agg_row["p50"], "p75": agg_row["p75"], "p90": agg_row["p90"],
    "p95": agg_row["p95"], "p99": agg_row["p99"],
    "max_score": agg_row["max_score"],
    "score_gt_001_count": int(agg_row["score_gt_001_count"]),
    "score_gt_01_count": int(agg_row["score_gt_01_count"]),
    "score_gt_05_count": int(agg_row["score_gt_05_count"]),
    "source": "epss_official",
    "batch_id": batch_id,
    "computed_at": INGESTED_AT,
    "is_systemic": is_systemic,
    "systemic_reason": "; ".join(reasons) if reasons else None,
}
dist_df = spark.createDataFrame([dist_row]).withColumn(
    "distribution_hash",
    content_hash([F.col("mean_score"), F.col("median_score"), F.col("p90"), F.col("p95"), F.col("p99"),
                  F.col("score_gt_01_count"), F.col("score_gt_05_count")]),
)

if table_exists(TBL["epss_distribution"]):
    target = DeltaTable.forName(spark, TBL["epss_distribution"])
    (
        target.alias("t")
        .merge(dist_df.alias("s"), "t.snapshot_date = s.snapshot_date")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    write_overwrite(dist_df, TBL["epss_distribution"])

log_batch_run("build_epss_distribution", "SUCCESS", record_count=1)
log_dq_metric("build_epss_distribution", "is_systemic", is_systemic, passed=True,
              detail=dist_row["systemic_reason"])
print(f"[build_epss_distribution] date={resolved_snapshot_date} is_systemic={is_systemic} "
      f"reason={dist_row['systemic_reason']} batch_id={batch_id}")
